In [ ]:
#Imports
import pickle
import os
from pathlib import Path
from shared.utils import run_model, load_snapshot
from shared.plotting import plot_sankey
from energyscope.models import Model

In [ ]:
# Définition du dossier
OUTPUT_DIR = '../03_Results'
# Chargement
with open(os.path.join(OUTPUT_DIR, 'results_wo_peaks_2050_update.pkl'), 'rb') as f:
    results_wo_peaks_2050 = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, 'results_wo_peaks_2050_carboneutre.pkl'), 'rb') as f:
    results_wo_peaks_2050_carboneutre = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, 'results_wo_peaks_2050_carboneutre_planHQ.pkl'), 'rb') as f:
    results_wo_peaks_2050_carboneutre_planHQ = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, 'results_peaks_2050_carboneutre.pkl'), 'rb') as f:
    results_peaks_2050_carboneutre = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, 'results_peaks_2050_carboneutre_planHQ.pkl'), 'rb') as f:
    results_peaks_2050_carboneutre_planHQ = pickle.load(f)


In [ ]:
from pathlib import Path

# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_UTILITIES_DIR = _BASE_DIR / 'shared' / 'utilities' / 'ES_snapshot'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle 2050 Carboneutre natif
energyscope_peaks_2050_carboneutre = Model([
    # --- STRUCTURES MATHÉMATIQUES (.mod) ---
    ('mod', str(_UTILITIES_DIR / 'QC_es_main.mod')),
    ('mod', str(_UTILITIES_DIR / 'QC_objective_function.mod')),
    ('mod', str(_PROJECT_DIR / 'HP_extra.mod')),
    ('mod', str(_PROJECT_DIR / 'peaks_extra.mod')),  # Contient la déclaration 'set PEAK_PERIODS;'
    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),
    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')), # Limite annuelle d'hydro
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.mod')),

    # --- DONNÉES DE BASE ENERGYSCOPE ---
    ('dat', str(_UTILITIES_DIR / 'QC_data.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_techs_dist_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_techs_B2D.dat')),
    ('dat', str(_UTILITIES_DIR / 'QC_mob_params.dat')),

    # --- SCÉNARIO 2050 CARBONEUTRE ---
    ('dat', str(_PROJECT_DIR / 'carboneutre.dat')),   # param co2_limit := 0;

    # --- ENCLENCHEMENT DU MODE POINTES (La feinte du 'let') ---
    # 1. Initialisation du set avec les périodes 13 et 14
    ('dat', str(_PROJECT_DIR / 'peaks_2_periods.dat')),

    # 2. Flag 'mod' pour basculer AMPL en mode script (fusionne {13, 14} dans PERIODS)
    ('mod', str(_PROJECT_DIR / 'peaks_extra.dat')),

    # 3. Injection des données 2050 (AMPL acceptera le 'let' à la ligne 4 car il est en mode script)
    ('dat', str(_PROJECT_DIR / 'peaks_2_2050.dat')),
    ('dat', str(_PROJECT_DIR / 'peaks_cpt.dat')),
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.dat')),
])

In [ ]:
import copy
import pickle
import numpy as np
from pathlib import Path

# --- CONFIGURATION DES CHEMINS D'ACCÈS ---
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
OUTPUT_DIR = _BASE_DIR / 'projects' / 'peaks' / '03_Results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True) # S'assurer que le dossier existe

# --- 1) INITIALISATION DE L'ANALYSE DE SENSIBILITÉ ---
# Définition de vos facteurs de sensibilité (ex: de -20% à +40% de la taille de la pointe initiale)
sensitivity_factors = [0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]

# Dictionnaire pour stocker tous les résultats de la sensibilité
sensitivity_results = {}

# Périodes sur lesquelles on souhaite appliquer la sensibilité de la pointe
peak_periods = [13, 14]

print("Initialisation de l'instance d'optimisation AMPL...")
# On compile/construit le modèle une seule fois pour économiser la mémoire et le temps
# (Comme fait après RUN THE MODEL dans le dépôt EnergyScope)
model_instance = energyscope_peaks_2050_carboneutre.build()

# Extraction des valeurs initiales de la demande pour pouvoir réinitialiser à chaque itération
# 'end_uses_demand' est une table indexée par [EUD_TYPE, PERIODS]
initial_demand = model_instance.parameters['end_uses_demand'].get_values()

# --- 2) BOUCLE D'OPTIMISATION DE LA SENSIBILITÉ ---
for factor in sensitivity_factors:
    print(f"\n" + "="*50)
    print(f"ÉXÉCUTION DU SCÉNARIO : POINTE x {factor:.1f}")
    print("="*50)

    # Étape critique : On crée une copie propre de la demande initiale pour ce run
    updated_demand = initial_demand.copy()

    # Modifier la demande uniquement pour les périodes de pointe (13 et 14)
    # Note : end_uses_demand a un index multi-niveau (EUD_TYPE, PERIOD)
    for index, value in initial_demand.items():
        eud_type, period = index
        if period in peak_periods:
            # On applique le facteur multiplicateur sur la demande électrique de pointe
            updated_demand[index] = value * factor

    # Injection des nouvelles demandes modifiées dans l'instance AMPL active
    model_instance.parameters['end_uses_demand'].set_values(updated_demand)

    # Lancement du solveur (généralement CPLEX ou Gurobi via AMPL)
    print(f"Lancement du solveur pour le facteur {factor}...")
    model_instance.solve()

    # Extraction et stockage des résultats sous forme d'un objet EnergyScope Results
    # (Utilise la méthode native d'EnergyScope pour capturer les variables et paramètres résolus)
    # On utilise copy.deepcopy() par sécurité pour figer l'état des résultats dans notre dictionnaire
    print(f"Sauvegarde locale des résultats pour le facteur {factor}...")
    sensitivity_results[factor] = copy.deepcopy(model_instance.get_results())

# --- 3) SAUVEGARDE DES RÉSULTATS SUR LE DISQUE ---
output_file = OUTPUT_DIR / 'sensitivity_peak_size_2050.pkl'
print(f"\nSauvegarde finale de tous les scénarios dans : {output_file}")

with open(output_file, 'wb') as f:
    pickle.dump(sensitivity_results, f)

print("Analyse de sensibilité terminée avec succès ! 🎉")